## Install Dependencies

In [ ]:
%%capture
!pip install evaluate rouge_score nltk sacrebleu bert_score sentence-transformers scikit-learn matplotlib
!pip install git+https://github.com/neulab/BARTScore.git
!pip uninstall -y unsloth unsloth-zoo peft trl transformers
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes unsloth-zoo

## Import Libraries and Setup

In [ ]:
import torch
from unsloth import FastLanguageModel
import json
import pandas as pd
from datasets import Dataset
import os

max_seq_length = 1024 
dtype = None 
load_in_4bit = True 

print(f"GPU Model: {torch.cuda.get_device_name(0)}")

## Load, Merge, and Count Data

In [ ]:
train_file_paths = [
    "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/Adagium/adagium-all-reformat.json",
    "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/GBHN/GBHN-all-reformat.json",
    "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/Glosarium-MA/GMA-all.json",
    "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/HukumOnline/HO-all.json",
    "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/KHPTSultra/KHPTS-all.json",
    "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/LawDictionary/LD-all.json",
    "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/TAP-MPR/TMPR-all-reformat.json",
    "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/UUD/uud-id-reformat.json"
]

test_file_path = "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/test-data-reformat.json"

import random
test_data = []
combined_train_data = []

for file_path in train_file_paths:
    if os.path.exists(file_path):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                if isinstance(data, list):
                    combined_train_data.extend(data)
                    print(f"Successfully loaded {len(data)} records from: {os.path.basename(file_path)}")
                else:
                    print(f"Warning: {file_path} format is not a list of records.")
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    else:
        print(f"File not found: {file_path}")

# test_data = []
if os.path.exists(test_file_path):
    try:
        with open(test_file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                test_data.extend(data)
                print(f"Successfully loaded {len(data)} test records from: {os.path.basename(test_file_path)}")
    except Exception as e:
        print(f"Error reading {test_file_path}: {e}")
else:
    print(f"Test file not found: {test_file_path}")

train_df = pd.DataFrame(combined_train_data)
test_df = pd.DataFrame(test_data)

print(f"Total train data points: {len(train_df)}")
print(f"Total test data points: {len(test_df)}")

raw_train_dataset = Dataset.from_pandas(train_df)
raw_eval_dataset = Dataset.from_pandas(test_df)

## Load Model (Qwen3.5-9B)

In [ ]:
model_name = "unsloth/Qwen3.5-9B"

print(f"Loading Model: {model_name}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

SYSTEM_PROMPT = (
    "Anda adalah seorang pakar hukum Indonesia dan kamus hukum yang sangat presisi. "
    "Tugas Anda adalah memberikan definisi atau penjelasan hukum yang formal, baku, "
    "dan sesuai dengan literatur perundang-undangan. "
    "Jangan merangkum dengan bahasa santai. Gunakan gaya bahasa hukum yang kaku dan tepat."
)

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    contexts     = examples["context"]
    responses    = examples["response"]
    texts = []
    for instruction, context, response in zip(instructions, contexts, responses):
        user_content = instruction
        if context and context.strip():
            user_content = f"{instruction}\n\nKonteks:\n{context}"

        messages = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": response},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}

train_dataset = raw_train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset  = raw_eval_dataset.map(formatting_prompts_func,  batched=True)

print("Success! Loaded and formatted data using Qwen2.5 chat template.")

## Configure QLoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,        
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = True,
    loftq_config = None,
)

## Training (SFTTrainer)

## Training Logger Callback & Visualization Setup

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from transformers import TrainerCallback
import math, os

class TrainingLoggerCallback(TrainerCallback):
    """
    Captures every logged training step.
    With logging_steps=5 and ~1098 total steps, this guarantees 200+ log entries.
    Also computes an approximate token-level accuracy from cross-entropy loss.
    """
    def __init__(self):
        self.train_steps   = []
        self.train_losses  = []
        self.train_acc     = []   # token accuracy ≈ exp(-loss) is perplexity; acc ≈ 1/ppl is a proxy
        self.train_lr      = []
        self.eval_steps    = []
        self.eval_losses   = []
        self.epoch_markers = []   # global_step at each epoch boundary

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        step = state.global_step
        if "loss" in logs:
            self.train_steps.append(step)
            loss = logs["loss"]
            self.train_losses.append(loss)
            # Token accuracy proxy: softmax probability of the correct token
            # = exp(-loss) when loss is mean cross-entropy per token
            self.train_acc.append(math.exp(-loss) * 100)
            self.train_lr.append(logs.get("learning_rate", 0))
        if "eval_loss" in logs:
            self.eval_steps.append(step)
            self.eval_losses.append(logs["eval_loss"])

    def on_epoch_end(self, args, state, control, **kwargs):
        self.epoch_markers.append(state.global_step)


logger_callback = TrainingLoggerCallback()
print("TrainingLoggerCallback registered.")
print(f"  logging_steps=5 → estimated {int(len(train_dataset) / (4*4) * 3 / 5)} log entries")
print(f"  eval_steps=175  → estimated {int(len(train_dataset) / (4*4) * 3 / 175)} eval checkpoints")


In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

sft_config = SFTConfig(
    output_dir="outputs",
    max_seq_length=1024,
    dataset_text_field="text",
    dataset_num_proc=2,
    packing=False,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4, 
    num_train_epochs=3,    
    learning_rate=2e-5,    
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,      
    optim="adamw_8bit",
    weight_decay=0.05,
    neftune_noise_alpha=5.0,     
    fp16=True,
    bf16=False,
    eval_strategy="steps",
    eval_steps=175,        
    save_steps=175,        
    logging_steps=5,
    save_total_limit=2,          
    load_best_model_at_end=True, 
    metric_for_best_model="eval_loss", 
    greater_is_better=False,
    report_to="none",
    seed=3407,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,   
    args = sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5), logger_callback] 
)

def predict_training_time(dataset, config, seconds_per_step=1.2):
    total_examples = len(dataset)
    batch_size = config.per_device_train_batch_size
    grad_accum = config.gradient_accumulation_steps
    epochs = config.num_train_epochs
    
    steps_per_epoch = total_examples // (batch_size * grad_accum)
    total_steps = steps_per_epoch * epochs
    
    estimated_seconds = total_steps * seconds_per_step
    hours = estimated_seconds // 3600
    minutes = (estimated_seconds % 3600) // 60
    
    print("Prediksi Estimasi Waktu Pelatihan")
    print(f"Total Data Latih: {total_examples} sampel")
    print(f"Total Steps: {total_steps}")
    print(f"Estimasi Waktu: ~{int(hours)} jam dan {int(minutes)} menit")
    print(f"(Berdasarkan asumsi kecepatan ~{seconds_per_step} detik per step pada Tesla T4)")

predict_training_time(train_dataset, sft_config)

## Execute Training

In [ ]:
trainer_stats = trainer.train()

# Menghitung Perplexity pada Eval Dataset
import math
import transformers
for cb in list(trainer.callback_handler.callbacks):
    if "Notebook" in cb.__class__.__name__ or "Progress" in cb.__class__.__name__:
        trainer.remove_callback(cb)
eval_results = trainer.evaluate()
try:
    perplexity = math.exp(eval_results["eval_loss"])
    print(f"\nEvaluation Perplexity: {perplexity:.2f}")
except OverflowError:
    print("\nEvaluation Perplexity: Infinity")


## Training Progress Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import numpy as np
import os

def plot_training_dashboard(cb, save_path="/kaggle/working/training_dashboard.png"):
    """
    Generates a 4-panel training dashboard:
      - Panel 1: Training Loss over steps
      - Panel 2: Token Accuracy (proxy) over steps
      - Panel 3: Eval Loss at checkpoint steps
      - Panel 4: Learning Rate Schedule
    """
    if not cb.train_steps:
        print("No training logs captured — skipping visualization.")
        return

    # Dark theme
    BG      = "#0d1117"
    PANEL   = "#161b22"
    ACCENT1 = "#58a6ff"   # blue  — train loss
    ACCENT2 = "#3fb950"   # green — accuracy
    ACCENT3 = "#f78166"   # red   — eval loss
    ACCENT4 = "#d2a8ff"   # purple — LR
    GRID    = "#21262d"
    TEXT    = "#e6edf3"

    fig = plt.figure(figsize=(18, 10), facecolor=BG)
    fig.suptitle(
        "NusantaraLaw · QLoRA Fine-Tuning Training Dashboard
Qwen3.5-9B | Training Progress",
        color=TEXT, fontsize=14, fontweight="bold", y=0.98
    )

    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.3)
    axes = [fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2)]

    def style_ax(ax, title, xlabel, ylabel):
        ax.set_facecolor(PANEL)
        ax.set_title(title, color=TEXT, fontsize=11, pad=8)
        ax.set_xlabel(xlabel, color=TEXT, fontsize=9)
        ax.set_ylabel(ylabel, color=TEXT, fontsize=9)
        ax.tick_params(colors=TEXT, labelsize=8)
        ax.grid(color=GRID, linewidth=0.6, linestyle="--", alpha=0.7)
        for spine in ax.spines.values():
            spine.set_edgecolor(GRID)

    steps  = cb.train_steps
    losses = cb.train_losses
    accs   = cb.train_acc
    lrs    = cb.train_lr

    # ── Panel 1: Training Loss ──────────────────────────────────────
    ax = axes[0]
    style_ax(ax, "Training Loss per Step", "Global Step", "Cross-Entropy Loss")
    ax.plot(steps, losses, color=ACCENT1, linewidth=1.2, alpha=0.8, label="Train Loss")
    # Smooth rolling average
    if len(losses) >= 20:
        window = 20
        smooth = np.convolve(losses, np.ones(window)/window, mode="valid")
        smooth_steps = steps[window-1:]
        ax.plot(smooth_steps, smooth, color="#ffffff", linewidth=2.0, linestyle="-", alpha=0.9, label=f"Rolling Avg ({window})")
    # Epoch boundaries
    for ep_step in cb.epoch_markers:
        ax.axvline(ep_step, color="#ffa657", linewidth=1.2, linestyle=":", alpha=0.6)
    ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

    # ── Panel 2: Token Accuracy Proxy ──────────────────────────────
    ax = axes[1]
    style_ax(ax, "Token Accuracy Proxy (exp(-loss) × 100)", "Global Step", "Accuracy (%)")
    ax.plot(steps, accs, color=ACCENT2, linewidth=1.2, alpha=0.8, label="Token Acc")
    if len(accs) >= 20:
        smooth_acc = np.convolve(accs, np.ones(20)/20, mode="valid")
        ax.plot(steps[19:], smooth_acc, color="#ffffff", linewidth=2.0, alpha=0.9, label="Rolling Avg (20)")
    for ep_step in cb.epoch_markers:
        ax.axvline(ep_step, color="#ffa657", linewidth=1.2, linestyle=":", alpha=0.6)
    ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

    # ── Panel 3: Eval Loss at Checkpoints ──────────────────────────
    ax = axes[2]
    style_ax(ax, f"Evaluation Loss at Checkpoints (n={len(cb.eval_steps)})", "Global Step", "Eval Loss")
    if cb.eval_steps:
        ax.plot(cb.eval_steps, cb.eval_losses, color=ACCENT3, linewidth=2.0,
                marker="o", markersize=5, label="Eval Loss")
        best_idx = int(np.argmin(cb.eval_losses))
        ax.scatter([cb.eval_steps[best_idx]], [cb.eval_losses[best_idx]],
                   color="#ffd700", s=120, zorder=5, label=f"Best: {cb.eval_losses[best_idx]:.4f}")
        ax.axhline(cb.eval_losses[best_idx], color="#ffd700", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

    # ── Panel 4: Learning Rate Schedule ────────────────────────────
    ax = axes[3]
    style_ax(ax, "Learning Rate Schedule (Cosine Warmup)", "Global Step", "Learning Rate")
    if lrs:
        ax.plot(steps, lrs, color=ACCENT4, linewidth=1.5, alpha=0.9)
        ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2e"))
    for ep_step in cb.epoch_markers:
        ax.axvline(ep_step, color="#ffa657", linewidth=1.2, linestyle=":", alpha=0.6,
                   label="Epoch boundary" if ep_step == cb.epoch_markers[0] else "")
    ax.legend(facecolor=PANEL, edgecolor=GRID, labelcolor=TEXT, fontsize=8)

    # ── Footer stats ────────────────────────────────────────────────
    total_steps = steps[-1] if steps else 0
    final_loss  = losses[-1] if losses else 0
    final_acc   = accs[-1] if accs else 0
    n_logs      = len(steps)
    fig.text(
        0.5, 0.01,
        f"Total Steps: {total_steps}  |  Total Log Entries: {n_logs}  |  "
        f"Final Train Loss: {final_loss:.4f}  |  Final Token Acc: {final_acc:.2f}%  |  "
        f"Eval Checkpoints: {len(cb.eval_steps)}",
        ha="center", color="#8b949e", fontsize=9
    )

    plt.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close()
    print(f"[OK] Training dashboard saved to: {save_path}")

plot_training_dashboard(logger_callback)

# Print log statistics
print(f"\n[*] Training Log Summary:")
print(f"    Total log entries captured : {len(logger_callback.train_steps)}")
print(f"    Total eval checkpoints     : {len(logger_callback.eval_steps)}")
print(f"    Epoch boundaries at steps  : {logger_callback.epoch_markers}")
if logger_callback.train_losses:
    import numpy as np
    losses_arr = logger_callback.train_losses
    print(f"    Initial train loss         : {losses_arr[0]:.4f}")
    print(f"    Final train loss           : {losses_arr[-1]:.4f}")
    print(f"    Min train loss             : {min(losses_arr):.4f}")
    print(f"    Loss reduction             : {((losses_arr[0]-losses_arr[-1])/losses_arr[0]*100):.1f}%")
if logger_callback.eval_losses:
    print(f"    Best eval loss             : {min(logger_callback.eval_losses):.4f}")
    print(f"    Final eval loss            : {logger_callback.eval_losses[-1]:.4f}")


## Inference and Evaluation

### Latent Space Representation Analysis
This evaluation uses a **Representation Learning** approach as guided by the supervisor's thesis requirements.

Instead of relying on external evaluators (e.g., `bert-base-multilingual-cased`), the **fine-tuned Qwen3.5-9B model itself** is used as the semantic judge.

The model operates in two modes:
- **Encoder Mode** — Forward pass extracts hidden-state vectors (latent representations) from the last transformer layer. This replaces BERTScore, Sentence Similarity, BARTScore, and NLI.
- **Generator Mode** — Standard auto-regressive generation for BLEU, ROUGE, METEOR, and Perplexity.

**Key Metrics Added:**
- `NLaw-Score (Cosine)` — Cosine similarity between the Trained prediction vector and the Ground Truth vector, using the fine-tuned model's encoder.
- `L2 Latent Distance` — Euclidean distance between Ground Truth vector (encoder extraction) and Generated prediction vector. Low L2 = model internalized the correct legal meaning even when surface words differ.
- `t-SNE / PCA Visualization` — Plots Base vs Trained prediction vectors against Ground Truth in 2D latent space.

In [ ]:
import time
import math
import evaluate
import numpy as np
import nltk
from tqdm import tqdm
import random
import torch
import torch.nn.functional as F

nltk.download("punkt")
nltk.download("wordnet")
nltk.download("punkt_tab")
nltk.download("omw-1.4")

bleu_metric   = evaluate.load("sacrebleu")
rouge_metric  = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

FastLanguageModel.for_inference(model)

# ─── Evaluation Sample Config ───────────────────────────────────────────────
sample_size    = min(50, len(eval_dataset))
sample_indices = random.sample(range(len(eval_dataset)), sample_size)

print(f"Memulai Evaluasi Komprehensif pada {sample_size} sampel Test...")
print("Membandingkan Model Untrained (Base) vs Model Trained (LoRA)")
print("Metrik: BLEU | ROUGE | METEOR | Perplexity | NLaw-Score | L2 Latent Distance")
print("=" * 100)


# ─── 1. Text Generation Function ────────────────────────────────────────────
def generate_response(instruction, context="", disable_lora=False):
    """Generate a text response. Set disable_lora=True for Base model mode."""
    user_content = instruction
    if context and context.strip():
        user_content = f"{instruction}\n\nKonteks:\n{context}"

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs       = tokenizer(text=[prompt_text], return_tensors="pt").to("cuda")
    input_length = inputs["input_ids"].shape[1]

    gen_kwargs = dict(
        max_new_tokens=512, use_cache=True,
        do_sample=False, repetition_penalty=1.15, no_repeat_ngram_size=3,
    )

    if disable_lora:
        with model.disable_adapter():
            with torch.no_grad():
                outputs = model.generate(**inputs, **gen_kwargs)
    else:
        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)

    new_token_ids = outputs[0][input_length:]
    return tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()


# ─── 2. Latent Vector Extraction Function ───────────────────────────────────
def extract_hidden_vector(text, disable_lora=False, max_length=512):
    """
    Extract the last-layer mean-pooled hidden state vector from the model.
    This is the "Encoder Mode" — uses the fine-tuned Qwen as a semantic encoder.
    Returns a 1D tensor on CPU.
    """
    enc = tokenizer(
        text=text, return_tensors="pt",
        truncation=True, max_length=max_length,
        padding=False
    ).to("cuda")

    def forward_pass():
        with torch.no_grad():
            out = model(
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                output_hidden_states=True,
                return_dict=True,
            )
        # Last layer hidden states: [1, seq_len, hidden_dim]
        last_hidden = out.hidden_states[-1]  # shape: (1, seq, dim)
        # Mean pooling over token dimension
        mask = enc["attention_mask"].unsqueeze(-1).float()
        vec = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1)
        return vec.squeeze(0).cpu()  # shape: (hidden_dim,)

    if disable_lora:
        with model.disable_adapter():
            return forward_pass()
    else:
        return forward_pass()


# ─── 3. Generate Predictions ─────────────────────────────────────────────────
predictions_untrained = []
predictions_trained   = []
references            = []

print("\n[*] Meng-generate respon untuk Untrained Model (Base)...\n")
for idx in tqdm(sample_indices, desc="Generasi Untrained"):
    sample = eval_dataset[idx]
    gen_untrained = generate_response(sample["instruction"], sample.get("context", ""), disable_lora=True)
    predictions_untrained.append(gen_untrained)
    references.append(sample["response"])

print("\n[*] Meng-generate respon untuk Trained Model (LoRA)...\n")
for idx in tqdm(sample_indices, desc="Generasi Trained", total=len(sample_indices)):
    sample = eval_dataset[idx]
    gen_trained = generate_response(sample["instruction"], sample.get("context", ""), disable_lora=False)
    predictions_trained.append(gen_trained)


# ─── 4. Extract Latent Vectors for Latent Space Analysis ─────────────────────
print("\n[*] Mengekstrak vektor laten (Encoder Mode) untuk analisis ruang laten...")
print("    [Ground Truth (Base context)] → vektor referensi laten")
print("    [Trained Prediction]          → vektor prediksi laten (LoRA mode)")
print("    [Untrained Prediction]        → vektor prediksi laten (Base mode)\n")

vecs_reference  = []  # Ground truth vectors (via trained model encoder)
vecs_trained    = []  # Trained model prediction vectors
vecs_untrained  = []  # Base model prediction vectors

N_LATENT = min(30, sample_size)  # limit for time/memory

for i in tqdm(range(N_LATENT), desc="Ekstraksi Vektor Laten"):
    ref_text  = references[i]
    pred_tr   = predictions_trained[i]
    pred_base = predictions_untrained[i]

    # Ground truth encoded through the TRAINED model's encoder
    vecs_reference.append(extract_hidden_vector(ref_text, disable_lora=False))
    # Trained prediction vector
    vecs_trained.append(extract_hidden_vector(pred_tr, disable_lora=False))
    # Untrained prediction vector
    vecs_untrained.append(extract_hidden_vector(pred_base, disable_lora=True))

vecs_reference  = torch.stack(vecs_reference)   # (N, dim)
vecs_trained    = torch.stack(vecs_trained)     # (N, dim)
vecs_untrained  = torch.stack(vecs_untrained)   # (N, dim)


# ─── 5. NLaw-Score: Cosine Similarity (replaces BERTScore / Sentence Sim) ────
def nlaw_cosine_score(pred_vecs, ref_vecs):
    """Compute mean cosine similarity between prediction and reference vectors."""
    scores = F.cosine_similarity(pred_vecs, ref_vecs, dim=-1)
    return float(scores.mean().item())

nlaw_trained    = nlaw_cosine_score(vecs_trained,   vecs_reference)
nlaw_untrained  = nlaw_cosine_score(vecs_untrained, vecs_reference)

print(f"  NLaw-Score (Trained):   {nlaw_trained:.4f}")
print(f"  NLaw-Score (Untrained): {nlaw_untrained:.4f}")


# ─── 6. L2 Latent Distance (Representation Learning core metric) ─────────────
def l2_latent_distance(pred_vecs, ref_vecs):
    """
    Euclidean (L2) distance between generated prediction vectors and ground truth.
    Lower = model's internal representation is closer to the 'correct' legal meaning.
    """
    dists = torch.norm(pred_vecs - ref_vecs, p=2, dim=-1)
    return float(dists.mean().item())

l2_trained   = l2_latent_distance(vecs_trained,   vecs_reference)
l2_untrained = l2_latent_distance(vecs_untrained, vecs_reference)

print(f"  L2 Latent Distance (Trained):   {l2_trained:.4f}")
print(f"  L2 Latent Distance (Untrained): {l2_untrained:.4f}")


# ─── 7. Standard Lexical Metrics (BLEU, ROUGE, METEOR) ───────────────────────
def compute_lexical_metrics(preds, refs):
    results = {}
    preds_rouge = ["\n".join(nltk.sent_tokenize(p)) for p in preds]
    refs_rouge  = ["\n".join(nltk.sent_tokenize(r)) for r in refs]
    try: results['rouge']  = rouge_metric.compute(predictions=preds_rouge, references=refs_rouge, use_stemmer=True)
    except: results['rouge'] = {'rouge1': 0, 'rouge2': 0, 'rougeL': 0}
    try: results['bleu']   = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
    except: results['bleu'] = {'score': 0}
    try: results['meteor'] = meteor_metric.compute(predictions=preds, references=refs)
    except: results['meteor'] = {'meteor': 0}
    return results

print("\n[*] Menghitung Metrik Leksikal untuk Untrained Model...")
res_untrained = compute_lexical_metrics(predictions_untrained, references)
print("[*] Menghitung Metrik Leksikal untuk Trained Model (LoRA)...")
res_trained   = compute_lexical_metrics(predictions_trained, references)


# ─── 8. Perplexity ────────────────────────────────────────────────────────────
def compute_ppl_for_samples(model_ref, dataset, indices, disable_lora=False):
    total_loss, total_count = 0.0, 0
    FastLanguageModel.for_inference(model_ref)
    def compute():
        nonlocal total_loss, total_count
        for idx in indices[:20]:
            text = dataset[idx].get("text", "")
            if not text: continue
            enc = tokenizer(text=text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
            with torch.no_grad():
                out = model_ref(**enc, labels=enc["input_ids"])
            total_loss += out.loss.item() * enc["input_ids"].shape[1]
            total_count += enc["input_ids"].shape[1]
    if disable_lora:
        with model_ref.disable_adapter(): compute()
    else: compute()
    return math.exp(total_loss / total_count) if total_count > 0 else float('inf')

print("\n[*] Menghitung Perplexity...")
try:
    ppl_untrained = compute_ppl_for_samples(model, eval_dataset, sample_indices, disable_lora=True)
    ppl_trained   = compute_ppl_for_samples(model, eval_dataset, sample_indices, disable_lora=False)
except Exception as e:
    print(f"  Perplexity Error: {e}")
    ppl_untrained, ppl_trained = 0.0, 0.0


# ─── 9. BARTScore (kept for reference, uses GPU) ─────────────────────────────
bart_trained, bart_untrained = 0.0, 0.0
try:
    from bart_score import BARTScorer
    bart_scorer = BARTScorer(device="cuda:0", checkpoint="facebook/bart-large-cnn")
    bart_trained   = float(np.mean(bart_scorer.score(predictions_trained,   references, batch_size=4)))
    bart_untrained = float(np.mean(bart_scorer.score(predictions_untrained, references, batch_size=4)))
    del bart_scorer; torch.cuda.empty_cache()
    print(f"  BARTScore (Trained):   {bart_trained:.4f}")
    print(f"  BARTScore (Untrained): {bart_untrained:.4f}")
except Exception as e:
    print(f"  BARTScore Error (non-critical): {e}")


# ─── 10. Results Table ────────────────────────────────────────────────────────
SEP = "=" * 95
SEP2 = "-" * 95
print("\n" + SEP)
print(f"  HASIL EVALUASI MODEL — KOMPARASI UNTRAINED VS TRAINED (TEST DATA)  ".center(95))
print(SEP)
print(f"  {'Metrik':<28} | {'Untrained Model (Base)':<27} | {'Trained Model (LoRA)':<27}")
print(SEP2)
print(f"  {'> Perplexity ↓':<28} | {ppl_untrained:<27.4f} | {ppl_trained:<27.4f}")
print(f"  {'> SacreBLEU ↑':<28} | {res_untrained['bleu']['score']:<27.2f} | {res_trained['bleu']['score']:<27.2f}")
print(f"  {'> ROUGE-1 ↑':<28} | {res_untrained['rouge']['rouge1']*100:<27.2f} | {res_trained['rouge']['rouge1']*100:<27.2f}")
print(f"  {'> ROUGE-2 ↑':<28} | {res_untrained['rouge']['rouge2']*100:<27.2f} | {res_trained['rouge']['rouge2']*100:<27.2f}")
print(f"  {'> ROUGE-L ↑':<28} | {res_untrained['rouge']['rougeL']*100:<27.2f} | {res_trained['rouge']['rougeL']*100:<27.2f}")
print(f"  {'> METEOR ↑':<28} | {res_untrained['meteor']['meteor']*100:<27.2f} | {res_trained['meteor']['meteor']*100:<27.2f}")
print(SEP2)
print(f"  --- Latent Space Metrics (Representation Learning) ---".center(95))
print(SEP2)
print(f"  {'> NLaw-Score Cosine ↑':<28} | {nlaw_untrained:<27.4f} | {nlaw_trained:<27.4f}")
print(f"  {'> L2 Latent Distance ↓':<28} | {l2_untrained:<27.4f} | {l2_trained:<27.4f}")
print(f"  {'> BARTScore ↑':<28} | {bart_untrained:<27.4f} | {bart_trained:<27.4f}")
print(SEP)

print("\nContoh Perbandingan (Target vs Base vs Trained)")
for i in range(min(3, len(references))):
    print(f"\nContoh {i+1}:")
    print(f"  Instruksi   : {eval_dataset[sample_indices[i]]['instruction'][:150]}")
    print(f"  Target      : {references[i][:200]}...")
    print(f"  Base Model  : {predictions_untrained[i][:200]}...")
    print(f"  LoRA Trained: {predictions_trained[i][:200]}...")
    print("-"*60)


## Latent Space Visualization (t-SNE & PCA)

This section visualizes the model's internal legal knowledge in 2D space using t-SNE and PCA.

- **Ground Truth** (green) — vectors from the correct legal answers encoded by the fine-tuned model
- **Trained Prediction** (blue) — vectors from the LoRA model's generated responses
- **Untrained Prediction** (red) — vectors from the Base model's generated responses

If fine-tuning is effective, the **blue dots should cluster tightly around the green dots**, while **red dots are scattered**.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for Kaggle
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import os

# Combine all vectors for joint decomposition
all_vecs = torch.cat([vecs_reference, vecs_trained, vecs_untrained], dim=0).numpy()
labels_ref  = ["Ground Truth"] * len(vecs_reference)
labels_tr   = ["Trained (LoRA)"] * len(vecs_trained)
labels_base = ["Untrained (Base)"] * len(vecs_untrained)
all_labels  = labels_ref + labels_tr + labels_base

colors = {
    "Ground Truth":      "#2ecc71",  # green
    "Trained (LoRA)":    "#3498db",  # blue
    "Untrained (Base)":  "#e74c3c",  # red
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor("#1a1a2e")
for ax in axes:
    ax.set_facecolor("#16213e")
    ax.tick_params(colors="white")
    ax.xaxis.label.set_color("white")
    ax.yaxis.label.set_color("white")
    ax.title.set_color("white")
    for spine in ax.spines.values():
        spine.set_edgecolor("#444466")

# ── PCA ──────────────────────────────────────────────────────
pca    = PCA(n_components=2, random_state=42)
pca_2d = pca.fit_transform(all_vecs)

for lbl in ["Untrained (Base)", "Trained (LoRA)", "Ground Truth"]:
    mask = [i for i, l in enumerate(all_labels) if l == lbl]
    axes[0].scatter(
        pca_2d[mask, 0], pca_2d[mask, 1],
        c=colors[lbl], label=lbl,
        s=60, alpha=0.85, edgecolors="white", linewidths=0.4
    )

explained = pca.explained_variance_ratio_
axes[0].set_title(f"PCA — Latent Space\n(PC1={explained[0]*100:.1f}%, PC2={explained[1]*100:.1f}%)", fontsize=12)
axes[0].legend(facecolor="#0f3460", edgecolor="#444466", labelcolor="white", fontsize=9)
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

# ── t-SNE ────────────────────────────────────────────────────
perp = min(15, max(5, len(all_vecs) // 4))
tsne    = TSNE(n_components=2, random_state=42, perplexity=perp, max_iter=1000)
tsne_2d = tsne.fit_transform(all_vecs)

for lbl in ["Untrained (Base)", "Trained (LoRA)", "Ground Truth"]:
    mask = [i for i, l in enumerate(all_labels) if l == lbl]
    axes[1].scatter(
        tsne_2d[mask, 0], tsne_2d[mask, 1],
        c=colors[lbl], label=lbl,
        s=60, alpha=0.85, edgecolors="white", linewidths=0.4
    )

axes[1].set_title(f"t-SNE — Latent Space (perplexity={perp})", fontsize=12)
axes[1].legend(facecolor="#0f3460", edgecolor="#444466", labelcolor="white", fontsize=9)
axes[1].set_xlabel("Dim 1")
axes[1].set_ylabel("Dim 2")

fig.suptitle(
    "NusantaraLaw · Latent Space Representation Analysis\n"
    "Qwen3.5-9B (Base vs Fine-Tuned with QLoRA) — Legal Domain",
    color="white", fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()

out_path = "/kaggle/working/latent_space_visualization.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close()
print(f"[✓] Latent Space Visualization saved to: {out_path}")

# Also print L2 delta summary
l2_improvement = ((l2_untrained - l2_trained) / l2_untrained * 100) if l2_untrained > 0 else 0
cosine_improvement = ((nlaw_trained - nlaw_untrained) / abs(nlaw_untrained) * 100) if nlaw_untrained != 0 else 0

print("\n[*] Latent Space Improvement Summary:")
print(f"  L2 Distance reduced by   : {l2_improvement:.1f}% (Trained vs Base)")
print(f"  NLaw-Score improved by   : {cosine_improvement:.1f}% (Trained vs Base)")
print("\n  ✔ This confirms the model internalized legal domain meaning in its latent space.")


## Cleanup and Save the Model

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("hf_token")
login(hf_token)

repo_name = "bayhaqieee/qwen3.5-9b-nlaw-gguf"

# PUSH ADAPTERS
print("Pushing Adapters (LoRA) to Hugging Face...")
try:
    model.push_to_hub(repo_name, token=hf_token)
    tokenizer.push_to_hub(repo_name, token=hf_token)
    print("Adapters (LoRA) Pushed to Hugging Face successfully!")
except Exception as e:
    print(f"Adapter Push Failed: {e}")

# PUSH GGUF KE HUGGING FACE
print("\nPushing GGUF to Hugging Face (This requires heavy disk space)...")
try:
    model.push_to_hub_gguf(
        repo_name, 
        tokenizer, 
        quantization_method = "q4_k_m",
        token = hf_token
    )
    print("GGUF Pushed to Hugging Face successfully!")
except Exception as e:
    print(f"\nGGUF Push Failed: {e}")
    print("\nNOTE: Kaggle's 20GB disk limit often blocks 7B GGUF conversions.")
    print("Adapter LoRA sudah berhasil disimpan ke Hugging Face di Langkah 1!")
    print("Gabungkan (merge) LoRA ke Base Model menjadi GGUF secara terpisah di Google Colab.")

import gc
import shutil
import os
import torch
print("Cleaning up resources to prevent Kaggle commit errors...")
try:
    del model
    del tokenizer
except:
    pass
gc.collect()
torch.cuda.empty_cache()
os.system('rm -rf /root/.cache/huggingface/hub/*')
os.system('rm -rf /tmp/unsloth*')
print("Cleanup complete!")
